# MITgcm preprocessing
Builds a ready-to-compile MITgcm run folder in `LZT_datafiles/2-preprocessing-outputs/<SimulationName>`.
All settings come from the `CurrentProject` block of `config.json`; data paths are resolved through `Config.Paths`.

**Steps**
1. Setup: load the config
2. Prepare the output folder from the templates of `00-template_mitgcm/`
3. Grid and bathymetry
4. Surface forcings (Alplakes weather → `binary_data/*.bin`)
5. Initial conditions (Simstrat profile or pickup)
6. Fill the template files (`run_config/data*`, `code/swfrac.F`, `postprocessing/*.py`)
7. Domain decomposition (`code/SIZE.h`, exch2)
8. Rivers (optional)
9. Launch script for kball_laken

**TODO:** choice of the pickup file, of the pickup and diagnostic frequencies, and of the output type (mnc or not).

## 1. Setup
The kernel must run from `preprocessing-mitgcm/` (plain imports of the library packages).

In [ ]:
from lib.config import ConfigObject
from lib.grid_and_bathy import get_dz_grid, get_grid
from lib.initial_conditions import build_initial_conditions
from lib.rivers import build_river_config, build_river_binaries
from lib.run_folder import (prepare_output_folder, copy_grid_and_bathy, get_run_times, get_mitgcm_results_folder,
                               fill_run_config_templates, write_data_config_files, write_secchi,
                               write_size_config_files, write_exch2_config, write_launch_script)
from lib.surface_forcings import download_weather, extract_and_save_surface_forcings

import os
import numpy as np

In [ ]:
Config = ConfigObject('config.json')

# Start and end of the run, in seconds from the reference date (MITgcm time axis)
StartTimeInSecond, EndTimeInSecond = get_run_times(Config)

print(f'Simulation: {Config.SimulationName} ({Config.LakeName}, grid {Config.GridConfigName})')
print(f'Period: {Config.StartDate} -> {Config.EndDate} (reference {Config.ReferenceDate})')
print(f'Templates: {Config.TemplateFolder} | init: {Config.InitializationType} | rivers: {Config.WithRivers}')

## 2. Prepare the output folder
Wipes `2-preprocessing-outputs/<SimulationName>/` (other simulations are kept), overlays the template folders listed in `TemplateFolder` in order, and writes `metadata_simulation.txt`.

In [ ]:
OutputFolder = prepare_output_folder(Config)
RunConfigFolder = os.path.join(OutputFolder, 'run_config')
BinaryDataFolder = os.path.join(OutputFolder, 'binary_data')
print(f'Output folder: {OutputFolder}')

## 3. Grid and bathymetry
Grid and bathymetry are created beforehand with the notebooks of `grid_and_bathy/` (`create_grid`, `create_bathy`). They are copied to `grid/` and `binary_data/bathy.bin`.

In [ ]:
copy_grid_and_bathy(Config, OutputFolder)

Grid = get_grid(Config.Paths.GridFolderPath)
DzGrid = get_dz_grid(os.path.join(Config.Paths.GridFolderPath, 'dz.csv'))
print(f'Grid: {Config.Nx} x {Config.Ny} cells, {np.count_nonzero(~np.isnan(DzGrid))} vertical layers')

## 4. Surface forcings
Downloads the missing weather days from the Alplakes API into `1-preprocessing-inputs/raw_from_alplakes_api/<lake>` (days already there are skipped), then interpolates them to the grid and writes the forcing binaries (big-endian `>f8`) to `binary_data/`.

In [ ]:
download_weather(Config, Grid)

In [ ]:
extract_and_save_surface_forcings(BinaryDataFolder, Config.StartDate, Config.EndDate, Config.Paths.RawWeatherFolder,
                                  Grid, Config.ALw, Config.WeatherModelType)

## 5. Initial conditions
Depends on `InitializationType`:
- `simstrat`: Simstrat temperature profile closest to the start date (downloaded into `1-preprocessing-inputs/simstrat_data/<lake>` if missing)
- `pickup`: restart from a MITgcm pickup file; its number is the number of time steps since the reference date

Salinity is a constant default.

In [ ]:
InitialTemperature, InitialSalt, PickupNumber = build_initial_conditions(Config, DzGrid, StartTimeInSecond)

## 6. Fill the template files
Replaces the `!placeholder!` tokens in the copied files only (never in `00-template_mitgcm/`):
- `run_config/data.cal`, `data.exf`, `data.diagnostics`, `data.mnc` and `postprocessing/*.py`: dates, forcing parameters, results folders
- `code/swfrac.F`: Secchi depths (light penetration)
- `run_config/data`: run parameters, initial conditions, vertical grid

In [ ]:
# Results folder on the computer running MITgcm
MitgcmResultsFolder = get_mitgcm_results_folder(Config)

fill_run_config_templates(Config, OutputFolder, MitgcmResultsFolder)
write_secchi(os.path.join(OutputFolder, 'code/swfrac.F'), Config.Secchi)
write_data_config_files(os.path.join(RunConfigFolder, 'data'), InitialTemperature, InitialSalt,
                        StartTimeInSecond, EndTimeInSecond, PickupNumber, DzGrid, Config)
print(f'Results folder on {Config.ComputerName}: {MitgcmResultsFolder}')

## 7. Domain decomposition
With the `exch2` template, the tiles that contain only land are dropped: the land tiles are read from `land_cores_Px{Px}_Py{Py}.txt` (written by `grid_and_bathy/check_core_allocation.ipynb`), and the remaining cores are laid out as a single row (`Py = 1`).

In [ ]:
UseExch2 = 'exch2' in Config.TemplateFolder
Px, Py = write_exch2_config(OutputFolder, Config) if UseExch2 else (Config.Px, Config.Py)

# Tile size (sNx, sNy) comes from the full decomposition of config.json, before land tiles are dropped
write_size_config_files(os.path.join(OutputFolder, 'code/SIZE.h'), Px, Py, Config.Nx, Config.Ny,
                        np.count_nonzero(~np.isnan(DzGrid)), int(Config.Nx / Config.Px), int(Config.Ny / Config.Py))
print(f'Using exch2: {UseExch2} | number of cores: {Px * Py}')

## 8. Rivers (optional)
Rivers enter as open boundaries (only if `WithRivers` is true).

In [ ]:
if Config.WithRivers:
    RiverDicts, RiverString = build_river_config(Config, RunConfigFolder)
    build_river_binaries(Config, BinaryDataFolder)
    print(RiverString)
else:
    print('No rivers')

## 9. Launch script
Writes `launch_simulation.sh`, which compiles and starts the simulation on the remote computer in one command (see "Next steps").

In [ ]:
LaunchScriptPath = write_launch_script(OutputFolder, Config, Px * Py, MitgcmResultsFolder)
print(f'Written: {LaunchScriptPath}')

## Next steps

1. Copy the simulation folder `LZT_datafiles/2-preprocessing-outputs/<SimulationName>` to `<root>/2-preprocessing-outputs/` on kball_laken (e.g. `bcollin/`; e.g. with `scp -r` or WinSCP).
2. On kball_laken, from `<root>` (the folder containing `2-preprocessing-outputs/`), run:

```bash
bash 2-preprocessing-outputs/<SimulationName>/launch_simulation.sh
```

The script compiles MITgcm (`genmake2`, `make`), prepares `run/`, creates the results folders (`<root>/3-MITgcm-outputs/<SimulationName>/`, from `ComputerConfig.ResultsFolder`) and starts the simulation with `mpirun` in the background. The MITgcm folder and build options are set in `config.json` (`ComputerConfig`); the MITgcm folder can be overridden with `MITGCM_ROOTDIR=/path/to/MITgcm bash launch_simulation.sh`.